# 03 — LSTM Temporal para Score de Pré-Violência

O YOLOv8 analisa cada frame de forma **independente**. Este notebook treina um módulo temporal que observa uma **janela de N frames consecutivos** e gera um score contínuo de probabilidade de violência iminente.

**Arquitetura:**
```
Features por frame (5 valores)
        ↓
    LSTM BiDir
        ↓
  Multi-Head Attention
        ↓
    Pooling + MLP
        ↓
 [pre_violence_score, violence_score]  ← saída entre 0 e 1
```

**Features de entrada por frame:**

| # | Feature | Descrição |
|---|---|---|
| 0 | violence_flag | 1.0 se YOLOv8 detectou violence neste frame |
| 1 | pre_violence_flag | 1.0 se YOLOv8 detectou pre_violence |
| 2 | aggression_score | Score de postura agressiva (MediaPipe) |
| 3 | person_density | Nº de pessoas normalizado |
| 4 | optical_flow_mag | Magnitude de movimento (placeholder) |

## 1. Instalação

In [ ]:
!pip install -q torch torchvision matplotlib seaborn scikit-learn

## 2. Imports

In [ ]:
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    confusion_matrix, precision_recall_curve
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
META_DIR   = Path("../data/metadata")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device   : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

## 3. Configuração

In [ ]:
CFG = {
    "window_frames":  16,      # frames por sequência (janela temporal)
    "input_size":     5,       # features por frame
    "hidden_size":    256,
    "num_layers":     2,
    "dropout":        0.3,
    "num_heads":      4,       # Multi-Head Attention heads

    "epochs":         60,
    "batch_size":     32,
    "lr":             5e-4,
    "weight_decay":   1e-4,
    "patience":       10,      # early stopping

    "val_split":      0.15,
    "test_split":     0.10,

    # Pesos de classe para lidar com desbalanceamento
    # pre_violence e violence têm peso maior
    "pos_weight_pre_v": 2.0,
    "pos_weight_viol":  3.0,
}

FEATURES_PATH = META_DIR / "temporal_features.json"
LSTM_WEIGHTS  = MODELS_DIR / "temporal_lstm.pt"

assert FEATURES_PATH.exists(), (
    f"Features temporais não encontradas: {FEATURES_PATH}\n"
    "Execute o notebook 01_dataset_preparation.ipynb primeiro."
)

print(f"✓ Features: {FEATURES_PATH}")
print(f"  Janela: {CFG['window_frames']} frames | Hidden: {CFG['hidden_size']} | Epochs: {CFG['epochs']}")

## 4. Dataset temporal

In [ ]:
class TemporalDataset(Dataset):
    """
    Carrega sequências de features por vídeo.
    Cada item: tensor (T, 5) de features + target [pre_v, viol] binário.

    Labels:
      class 0 (person)       → pre_v=0, viol=0
      class 1 (violence)     → pre_v=1, viol=1
      class 2 (pre_violence) → pre_v=1, viol=0
    """

    def __init__(self, json_path: str, window: int = 16):
        with open(json_path) as f:
            raw = json.load(f)

        self.window  = window
        self.samples = []

        for item in raw:
            feats = np.array(item["features"], dtype=np.float32)  # (T, 5)
            label = int(item["label"])

            # Pad / crop para janela fixa
            T = feats.shape[0]
            if T < window:
                pad = np.zeros((window - T, feats.shape[1]), dtype=np.float32)
                feats = np.vstack([pad, feats])
            else:
                # Sliding window — gera múltiplas amostras por vídeo longo
                for start in range(0, T - window + 1, window // 2):
                    seg = feats[start:start + window]
                    pre_v = 1.0 if label >= 2 else 0.0
                    viol  = 1.0 if label == 1 else 0.0
                    self.samples.append((
                        torch.from_numpy(seg),
                        torch.tensor([pre_v, viol], dtype=torch.float32)
                    ))
                continue

            pre_v = 1.0 if label >= 2 else 0.0
            viol  = 1.0 if label == 1 else 0.0
            self.samples.append((
                torch.from_numpy(feats[:window]),
                torch.tensor([pre_v, viol], dtype=torch.float32)
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# Carregar e inspecionar
full_ds = TemporalDataset(str(FEATURES_PATH), window=CFG["window_frames"])
print(f"Total de sequências : {len(full_ds)}")

# Distribuição
targets = torch.stack([full_ds[i][1] for i in range(len(full_ds))])
n_viol     = int(targets[:, 1].sum())
n_pre_viol = int((targets[:, 0] == 1).sum()) - n_viol
n_normal   = len(full_ds) - n_viol - n_pre_viol
print(f"  normal       : {n_normal}")
print(f"  pre_violence : {n_pre_viol}")
print(f"  violence     : {n_viol}")

In [ ]:
# ── Split ─────────────────────────────────────────────────────────────────────
n_total = len(full_ds)
n_test  = max(1, int(n_total * CFG["test_split"]))
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val - n_test

train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False, num_workers=2)

print(f"Split — train: {n_train}  val: {n_val}  test: {n_test}")

## 5. Modelo LSTM com Attention

In [ ]:
class TemporalViolenceClassifier(nn.Module):
    """
    LSTM + Multi-Head Attention para classificação temporal de violência.

    Entrada : (batch, seq_len, input_size)
    Saída   : (batch, 2)  →  [pre_violence_score, violence_score]  ∈ [0,1]
    """

    def __init__(self, input_size=5, hidden_size=256, num_layers=2,
                 dropout=0.3, num_heads=4):
        super().__init__()

        # Projeção de entrada
        self.input_proj = nn.Sequential(
            nn.Linear(input_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
        )

        # LSTM
        self.lstm = nn.LSTM(
            input_size  = hidden_size // 2,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
            bidirectional = False,
        )

        # Attention para ponderar os timesteps mais relevantes
        self.attention = nn.MultiheadAttention(
            embed_dim   = hidden_size,
            num_heads   = num_heads,
            dropout     = dropout,
            batch_first = True,
        )

        # Classificador
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 32),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(32, 2),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, T, 5)
        x        = self.input_proj(x)              # (B, T, H/2)
        lstm_out, _ = self.lstm(x)                 # (B, T, H)
        attn_out, attn_weights = self.attention(
            lstm_out, lstm_out, lstm_out
        )                                           # (B, T, H)
        pooled = attn_out.mean(dim=1)              # (B, H)
        return self.classifier(pooled), attn_weights


model = TemporalViolenceClassifier(
    input_size  = CFG["input_size"],
    hidden_size = CFG["hidden_size"],
    num_layers  = CFG["num_layers"],
    dropout     = CFG["dropout"],
    num_heads   = CFG["num_heads"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Modelo: TemporalViolenceClassifier")
print(f"Parâmetros: {n_params:,}")
print(model)

## 6. Treinamento

In [ ]:
# ── Loss com pesos de classe (combate desbalanceamento) ────────────────────────
pos_weight = torch.tensor(
    [CFG["pos_weight_pre_v"], CFG["pos_weight_viol"]]
).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Como o modelo já usa Sigmoid na saída, usando BCELoss
criterion = nn.BCELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["epochs"], eta_min=1e-6
)


def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct_pre, correct_viol, total = 0.0, 0, 0, 0

    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred, _ = model(x)

            loss = criterion(pred, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss   += loss.item() * x.size(0)
            correct_pre  += ((pred[:, 0] > 0.5) == (y[:, 0] > 0.5)).sum().item()
            correct_viol += ((pred[:, 1] > 0.5) == (y[:, 1] > 0.5)).sum().item()
            total        += x.size(0)

    return total_loss / total, correct_pre / total, correct_viol / total


print("✓ Otimizador e loss configurados")

In [ ]:
# ── Loop de treinamento com early stopping ────────────────────────────────────
history = {
    "train_loss": [], "val_loss": [],
    "train_acc_prev": [], "val_acc_prev": [],
    "train_acc_viol": [], "val_acc_viol": [],
}

best_val_loss    = float("inf")
patience_counter = 0

print(f"{'Época':>6} | {'Train Loss':>10} | {'Val Loss':>9} | {'Acc PreV':>9} | {'Acc Viol':>9} | {'LR':>10}")
print("-" * 70)

for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()
    tr_loss, tr_acc_pre, tr_acc_viol = run_epoch(train_loader, train=True)
    vl_loss, vl_acc_pre, vl_acc_viol = run_epoch(val_loader,   train=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc_prev"].append(tr_acc_pre)
    history["val_acc_prev"].append(vl_acc_pre)
    history["train_acc_viol"].append(tr_acc_viol)
    history["val_acc_viol"].append(vl_acc_viol)

    lr_now = scheduler.get_last_lr()[0]

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>6} | {tr_loss:>10.4f} | {vl_loss:>9.4f} | "
              f"{vl_acc_pre:>9.3f} | {vl_acc_viol:>9.3f} | {lr_now:>10.2e}")

    # Early stopping + salvar melhor
    if vl_loss < best_val_loss:
        best_val_loss    = vl_loss
        patience_counter = 0
        torch.save(model.state_dict(), str(LSTM_WEIGHTS))
    else:
        patience_counter += 1
        if patience_counter >= CFG["patience"]:
            print(f"\nEarly stopping na época {epoch} (sem melhora por {CFG['patience']} épocas)")
            break

print(f"\n✓ Treinamento concluído. Melhor val_loss: {best_val_loss:.4f}")
print(f"  Pesos salvos em: {LSTM_WEIGHTS}")

## 7. Curvas de aprendizado

In [ ]:
epochs_ran = len(history["train_loss"])
x_axis     = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(x_axis, history["train_loss"], label="Train", color="steelblue")
axes[0].plot(x_axis, history["val_loss"],   label="Val",   color="tomato")
axes[0].set_title("Loss (BCE)", fontsize=12)
axes[0].set_xlabel("Época")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Acurácia Pre-Violence
axes[1].plot(x_axis, history["train_acc_prev"], label="Train", color="darkorange")
axes[1].plot(x_axis, history["val_acc_prev"],   label="Val",   color="goldenrod")
axes[1].set_title("Acurácia — pre_violence", fontsize=12)
axes[1].set_xlabel("Época")
axes[1].legend()
axes[1].grid(alpha=0.3)

# Acurácia Violence
axes[2].plot(x_axis, history["train_acc_viol"], label="Train", color="firebrick")
axes[2].plot(x_axis, history["val_acc_viol"],   label="Val",   color="tomato")
axes[2].set_title("Acurácia — violence", fontsize=12)
axes[2].set_xlabel("Época")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle("Curvas de Aprendizado — LSTM Temporal", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Avaliação no conjunto de teste

In [ ]:
# ── Carregar melhor modelo e avaliar ──────────────────────────────────────────
model.load_state_dict(torch.load(str(LSTM_WEIGHTS), map_location=DEVICE))
model.eval()

all_preds, all_targets, all_scores = [], [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        scores, _ = model(x)
        scores = scores.cpu().numpy()
        y      = y.numpy()

        # Violence label: violence_score > 0.5 OU pre_violence_score > 0.5
        pred_viol  = (scores[:, 1] > 0.5).astype(int)
        pred_prev  = (scores[:, 0] > 0.5).astype(int)
        true_viol  = (y[:, 1] > 0.5).astype(int)
        true_prev  = (y[:, 0] > 0.5).astype(int)

        all_preds.append(np.stack([pred_prev, pred_viol], axis=1))
        all_targets.append(y)
        all_scores.append(scores)

all_preds   = np.vstack(all_preds)
all_targets = np.vstack(all_targets)
all_scores  = np.vstack(all_scores)

print("── Relatório de Classificação (pre_violence) ──")
print(classification_report(
    all_targets[:, 0].astype(int),
    all_preds[:, 0],
    target_names=["normal", "pre_violence"]
))

print("── Relatório de Classificação (violence) ──")
print(classification_report(
    all_targets[:, 1].astype(int),
    all_preds[:, 1],
    target_names=["not_violence", "violence"]
))

In [ ]:
# ── Curvas ROC ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, idx, title, color in [
    (axes[0], 0, "Pre-Violence", "darkorange"),
    (axes[1], 1, "Violence",     "tomato"),
]:
    try:
        auc = roc_auc_score(all_targets[:, idx], all_scores[:, idx])
        fpr, tpr, _ = roc_curve(all_targets[:, idx], all_scores[:, idx])
        ax.plot(fpr, tpr, color=color, lw=2, label=f"AUC = {auc:.3f}")
        ax.plot([0,1], [0,1], "k--", lw=1)
        ax.set_title(f"ROC — {title}", fontsize=12)
        ax.set_xlabel("FPR")
        ax.set_ylabel("TPR")
        ax.legend()
        ax.grid(alpha=0.3)
    except Exception as e:
        ax.set_title(f"{title} — {e}")

plt.suptitle("Curvas ROC no Conjunto de Teste", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Matrizes de confusão ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, idx, title, labels in [
    (axes[0], 0, "Pre-Violence", ["normal", "pre_viol"]),
    (axes[1], 1, "Violence",     ["not_viol", "violence"]),
]:
    cm = confusion_matrix(all_targets[:, idx].astype(int), all_preds[:, idx])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f"Confusão — {title}", fontsize=11)
    ax.set_ylabel("Real")
    ax.set_xlabel("Predito")

plt.tight_layout()
plt.show()

## 9. Visualizar atenção temporal (interpretabilidade)

In [ ]:
# ── Visualizar pesos de atenção em sequências de exemplo ──────────────────────
# Mostra quais frames da sequência o modelo "prestou mais atenção"

model.eval()
sample_x, sample_y = next(iter(test_loader))
sample_x = sample_x[:4].to(DEVICE)

with torch.no_grad():
    preds, attn = model(sample_x)

# attn shape: (batch, heads, T, T) ou (batch, T, T)
# Colapsamos os heads e pegamos a linha de atenção média
if attn.dim() == 4:
    attn_avg = attn.mean(dim=1).cpu().numpy()   # (B, T, T)
else:
    attn_avg = attn.cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for i, ax in enumerate(axes):
    weights = attn_avg[i].mean(axis=0)   # atenção média sobre query → (T,)
    weights = weights / weights.sum()
    pre_v   = preds[i, 0].item()
    viol    = preds[i, 1].item()
    color   = "tomato" if viol > 0.5 else "darkorange" if pre_v > 0.5 else "steelblue"

    ax.bar(range(CFG["window_frames"]), weights, color=color, alpha=0.8)
    ax.set_title(f"PreV={pre_v:.2f}  Viol={viol:.2f}", fontsize=9, color=color)
    ax.set_xlabel("Frame na janela")
    ax.set_ylabel("Peso de atenção")
    ax.grid(alpha=0.3)

plt.suptitle("Pesos de Atenção por Frame (4 amostras)", fontsize=12)
plt.tight_layout()
plt.show()
print("Vermelho = violence detectada | Laranja = pre_violence | Azul = normal")

In [ ]:
# ── Resumo final ──────────────────────────────────────────────────────────────
try:
    auc_prev = roc_auc_score(all_targets[:, 0], all_scores[:, 0])
    auc_viol = roc_auc_score(all_targets[:, 1], all_scores[:, 1])
except:
    auc_prev = auc_viol = float("nan")

print("=" * 50)
print(" RESUMO DO LSTM TEMPORAL")
print("=" * 50)
print(f" Pesos salvos : {LSTM_WEIGHTS}")
print(f" AUC pre_viol : {auc_prev:.4f}")
print(f" AUC violence : {auc_viol:.4f}")
print(f" Janela       : {CFG['window_frames']} frames")
print("=" * 50)
print(" Próximo passo: 04_inference_demo.ipynb")